# Chapter 3 — Transformer Anatomy

This notebook is an annotated learning version of the Chapter 3 work from *Natural Language Processing with Transformers*.  
The goal of this chapter is to move from using pretrained transformer models as black boxes to understanding the main components that make an encoder-style transformer work.

In this chapter, the code walks through the internal building blocks behind BERT/DistilBERT-style models:

- tokenization and token IDs
- token embeddings
- scaled dot-product self-attention
- multi-head attention
- feed-forward networks
- residual connections and layer normalization
- positional embeddings
- stacking transformer encoder layers
- adding a sequence-classification head
- attention masks, including causal masks

The outputs are intentionally preserved. They show model structures, tensor shapes, parameter counts, attention matrices, and other intermediate results that are important for understanding how each block behaves.


## Setup: imports for building a transformer encoder

This notebook rebuilds the main parts of an encoder-style transformer using PyTorch.  
The imports include Hugging Face utilities for configuration/tokenization and PyTorch modules for implementing attention, feed-forward layers, embeddings, and classification heads.

In [1]:
# Imports
# Bring in math, Transformers, PyTorch, and neural network utilities.
from math import sqrt
import transformers
from transformers import AutoModel, AutoTokenizer, AutoConfig
import torch
from torch import nn
import torch.nn.functional as F

## Load DistilBERT components

This cell loads the DistilBERT tokenizer, base model, and configuration.  
The configuration is especially important because the manual implementation reuses values such as hidden size, number of layers, number of heads, vocabulary size, dropout, and maximum positions.

In [2]:
# Load DistilBERT
# Load tokenizer, model, config, and a small test sentence.
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)
config = AutoConfig.from_pretrained(model_ckpt)
text = "time flies like an arrow"
print(model)

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

## Tokenize a simple sentence

The example sentence is converted into token IDs without special tokens.  
Using a short sequence makes it easier to follow tensor shapes through each custom transformer component.

In [3]:
# Tokenize text
# Create token IDs for the example sentence without special tokens.
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
inputs.input_ids

tensor([[ 2051, 10029,  2066,  2019,  8612]])

## Count the parameters in the pretrained model

This cell sums the number of parameters in the loaded DistilBERT model.  
It gives a concrete sense of model scale before rebuilding a simplified version of its architecture.

In [4]:
# Parameter count
# Count model parameters to understand the size of the pretrained encoder.
x = sum(p.numel() for p in model.parameters())
print(x)

66362880


## Define scaled dot-product attention

Scaled dot-product attention is the core operation inside each attention head.  
It compares queries with keys, normalizes the scores with softmax, and uses the resulting weights to combine value vectors.

In [5]:
# Attention function
# Compute scaled dot-product attention from query, key, and value tensors.
def scaled_dot_product_attention(query, key, value):
    # Scale scores by the attention head dimension.
    dim_k = value.size(-1)
    # Compute token-token similarity with batched matrix multiplication.
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
    # Normalize scores into attention probabilities.
    weights = F.softmax(scores, dim=-1)
    attn = torch.bmm(weights, value)
    return attn

In [7]:
token_emb = nn.Embedding(config.vocab_size, config.dim)
input_embeds = token_emb(inputs.input_ids)
query = key = value = input_embeds
dim_k = key.size(-1)
scores = torch.bmm(query, key.transpose(1,2)) / sqrt(dim_k)
scores.size()

torch.Size([1, 5, 5])

## Implement a single attention head

An attention head learns separate linear projections for queries, keys, and values.  
The head receives hidden states and returns contextualized representations based on token-to-token interactions.

In [8]:
# Attention head
# Create learned Q/K/V projections for one attention head.
class AttentionHead(nn.Module):
    def __init__(self, emb_dim, head_dim):
        super().__init__()
        # Learned query, key, and value projections for this head.
        self.q = nn.Linear(emb_dim, head_dim)
        self.k = nn.Linear(emb_dim, head_dim)
        self.v = nn.Linear(emb_dim, head_dim)

    def forward(self, hidden_state):
        # Apply attention after projecting hidden states.
        attn_outputs = scaled_dot_product_attention(
            self.q(hidden_state), self.k(hidden_state), self.v(hidden_state)
        )
        return attn_outputs

## Implement multi-head attention

Multi-head attention creates several attention heads in parallel.  
Each head can specialize in a different type of relation, and the concatenated result is projected back into the original hidden dimension.

In [9]:
# Multi-head attention
# Combine several attention heads and project their concatenated output.
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        emb_dim = config.dim
        num_head = config.n_heads
        # Split the hidden dimension equally across heads.
        head_dim = emb_dim // num_head
        self.heads = nn.ModuleList(
            [AttentionHead(emb_dim, head_dim) for _ in range(num_head)]
        )
        self.linear_output = nn.Linear(head_dim*num_head, emb_dim) 
    def forward(self, hidden_state):
        # Join all head outputs before the final projection.
        x = torch.cat([h(hidden_state) for h in self.heads], dim=-1)
        x = self.linear_output(x)
        return x

## Implement the feed-forward network

After attention mixes information across tokens, each token is passed through the same feed-forward network independently.  
This block expands the hidden dimension, applies GELU nonlinearity, projects back down, and applies dropout.

In [10]:
# Feed-forward block
# Apply the transformer MLP block after attention.
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Expand hidden size to the intermediate feed-forward dimension.
        self.linear1 = nn.Linear(config.dim, config.hidden_dim)
        self.linear2 = nn.Linear(config.hidden_dim, config.dim)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # Position-wise MLP: every token is processed independently.
        x = self.linear1(x)
        x = self.gelu(x)
        x = self.linear2(x)
        x = self.dropout(x)
        return x

## Build one transformer encoder layer

A transformer encoder layer combines multi-head attention, residual connections, layer normalization, and the feed-forward network.  
This notebook uses the same high-level structure used by encoder-only models such as BERT and DistilBERT.

In [11]:
# Encoder layer
# Combine attention, residual connections, layer normalization, and feed-forward layers.
class TransformerEncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(config.dim)
        self.layer_norm2 = nn.LayerNorm(config.dim)
        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)
    def forward(self, x):
        hidden_state = self.layer_norm1(x)
        # Residual connection around the attention block.
        x = x + self.attention(hidden_state)
        # Residual connection around the feed-forward block.
        x = x + self.feed_forward(self.layer_norm2(x))
        return x

## Implement token and positional embeddings

Transformers do not process tokens in order by recurrence.  
So this embedding module adds token embeddings and positional embeddings, then normalizes and drops out the result before sending it into encoder layers.

In [12]:
# Embeddings
# Add token and positional embeddings before the encoder stack.
class Embeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Token IDs are mapped to dense vectors.
        self.token_embedding = nn.Embedding(
            config.vocab_size, config.dim
        )
        # Position IDs provide order information.
        self.positional_embedding = nn.Embedding(
            config.max_position_embeddings, config.dim
        )
        self.layer_norm = nn.LayerNorm(config.dim, eps=1e-12)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        seq_length = input_ids.size(1)
        postion_ids = torch.arange(seq_length, dtype=torch.long).unsqueeze(0)
        token_embedding = self.token_embedding(input_ids)
        positional_embedding = self.positional_embedding(postion_ids)
        embeddings = token_embedding + positional_embedding
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

## Test the embedding module

This cell checks that the embedding module returns the expected tensor shape.  
The shape should remain `[batch_size, sequence_length, hidden_size]`, which is the input format expected by encoder layers.

In [13]:
# Embedding test
# Check the output size of the embedding module.
embedding = Embeddings(config)
embedding(inputs.input_ids).size()

torch.Size([1, 5, 768])

## Inspect the embedding module

Printing the embedding module shows the exact PyTorch layers used for token embeddings, positional embeddings, layer normalization, and dropout.

In [14]:
# Embedding structure
# Print the embedding module to inspect its sublayers.
embedding

Embeddings(
  (token_embedding): Embedding(30522, 768)
  (positional_embedding): Embedding(512, 768)
  (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

## Stack encoder layers into a full encoder

A transformer encoder is created by stacking multiple encoder layers.  
The input first passes through embeddings, then through each layer sequentially, producing contextualized token representations.

In [15]:
# Encoder stack
# Stack multiple encoder layers into a full transformer encoder.
class TransformerEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embedding = Embeddings(config)
        self.layers = nn.ModuleList(
            [TransformerEncoderLayer(config) for _ in range(config.n_layers)]
        )

    def forward(self, x):
        # Convert token IDs to embeddings before the encoder stack.
        x = self.embedding(x)
        # Pass representations through each transformer block.
        for layer in self.layers:
            x = layer(x)

        return x

## Test the full encoder output shape

This cell sends token IDs through the custom encoder and checks the resulting tensor size.  
The output contains one contextualized vector for each token in the input sequence.

In [16]:
# Encoder test
# Check the output size of the full encoder.
encoder = TransformerEncoder(config)
encoder(inputs.input_ids).size()

torch.Size([1, 5, 768])

## Inspect the DistilBERT configuration

The configuration output shows the hyperparameters that control the architecture.  
This is useful for verifying that the custom modules match the pretrained model’s expected dimensions.

In [17]:
# Configuration
# Display the architecture hyperparameters used by DistilBERT.
config

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "transformers_version": "4.56.1",
  "vocab_size": 30522
}

## Add a sequence-classification head

For classification, the encoder output is reduced to a single vector and passed through a linear classifier.  
This mirrors the common pattern used by transformer models for tasks such as sentiment or emotion classification.

In [18]:
# Classification head
# Attach a classifier on top of the encoder output.
class TransformersForSequenceClassification(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encoder = TransformerEncoder(config)
        self.dropout = nn.Dropout(config.seq_classif_dropout)
        self.classifier = nn.Linear(config.dim, config.num_labels)
    def forward(self, x):
        # Use the first token representation as the sequence summary.
        x = self.encoder(x)[:, 0, :]
        x = self.dropout(x)
        x = self.classifier(x)
        return x

## Test the classifier output shape

This cell verifies that the classification model returns logits with shape `[batch_size, num_labels]`.  
That is the expected output format for sequence classification.

In [19]:
# Classifier test
# Check that the model returns one logit vector per input sequence.
encoder_classifier = TransformersForSequenceClassification(config)
encoder_classifier(inputs.input_ids).size()

torch.Size([1, 2])

## Get the sequence length for masking

This cell stores the sequence length so a mask can be built.  
Masks are used to control which tokens are allowed to attend to which other tokens.

In [20]:
# Sequence length
# Store the current input length for building an attention mask.
seq_len = inputs.input_ids.size(-1)

## Create a lower-triangular causal mask

A causal mask allows each position to attend only to itself and previous positions.  
This pattern is common in decoder-only language models, while encoder models usually use padding masks instead.

In [21]:
# Causal mask
# Build a lower-triangular mask for autoregressive attention.
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
mask[0]

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

## Apply the mask to attention scores

Masked positions are replaced with negative infinity before softmax.  
After softmax, these positions receive attention probability zero, preventing information flow from disallowed tokens.

In [22]:
# Masked scores
# Demonstrate how masked positions become impossible to attend to.
scores.masked_fill(mask == 0, -float("inf"))

tensor([[[ 2.8298e+01,        -inf,        -inf,        -inf,        -inf],
         [ 1.2173e+00,  2.8138e+01,        -inf,        -inf,        -inf],
         [-9.4138e-01,  1.3297e-01,  2.6670e+01,        -inf,        -inf],
         [-8.7914e-03,  3.4936e-01,  1.5686e+00,  2.9491e+01,        -inf],
         [ 2.0384e-01,  2.1633e+00,  2.3052e-01, -1.1465e+00,  2.6357e+01]]],
       grad_fn=<MaskedFillBackward0>)

## Extend attention with an optional mask

This final version of scaled dot-product attention supports masking.  
The same idea is used in practical transformer implementations for padding masks, causal masks, and other attention constraints.

In [23]:
# Masked attention
# Extend the attention function so masks can be applied optionally.
def scaled_dot_product_attention(query, key, value, mask=None):
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
    # Masking removes disallowed attention positions before softmax.
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights.bmm(value)